<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download
snapshot_download(repo_id="stronman/gemma-4-31B-it", local_dir="./gemma-31b-local")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

'/content/gemma-31b-local'

In [2]:
!pip install -q gradio

In [3]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

# 指向你刚才下载的本地文件夹
model_path = "./gemma-31b-local"

print("正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("正在满血加载 31B 模型到显存，请耐心等待 1-2 分钟...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("加载完成！准备启动网页服务...")

# 定义处理对话的核心函数
def predict(message, history):
    messages = []
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    streamer = TextIteratorStreamer(tokenizer, timeout=10.0, skip_prompt=True, skip_special_tokens=True)

    generate_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True
    )

    t = Thread(target=model.generate, kwargs=generate_kwargs)
    t.start()

    partial_message = ""
    for new_token in streamer:
        partial_message += new_token
        yield partial_message

# 创建 Gradio 聊天界面
demo = gr.ChatInterface(
    fn=predict,
    title="我的 Gemma-4-31B 专属大模型",
    description="运行在 Colab Pro A100 (96GB VRAM) 上的满血本地大模型！",
)

# 启动服务并生成公网链接
demo.launch(share=True, debug=True)

正在加载 Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


正在满血加载 31B 模型到显存，请耐心等待 1-2 分钟...


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


加载完成！准备启动网页服务...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ca39e2ef261174fdae.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ca39e2ef261174fdae.gradio.live
